# 03 — Composition over time, within lineages

---

**The core analysis.** Follows Figures 2 and 3 of the paper.

### Why fit inside lineages

A dataset-wide average can move for two different reasons:

1. the genomes inside each lineage are genuinely changing, or
2. nothing changes, but a lineage that already had a high value grows while one with a low value shrinks.

The second produces a confident-looking trend with no evolution behind it. Fitting **separately inside each lineage** removes it, which is what the paper does.

### Choosing the lineage level

Full GISAID labels are too fine — most span about 6 years with a few hundred genomes, and a slope's uncertainty grows as both span and sample size shrink. Truncating the label to **three levels** gives 9 well-populated lineages, close to the 12 the paper used.

---

In [ ]:
import sys, os, subprocess
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = os.path.abspath("..")          # repo root; the R scripts expect to run from here
pd.set_option("display.width", 130)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
PAL = {"A": "#2C7FB8", "B": "#C62824"}    # same colours as the R figures

def run_r(script="R/figures.R"):
    """Run the R figure script from the repo root and report what it wrote."""
    r = subprocess.run(["Rscript", script], capture_output=True, text=True, cwd=ROOT)
    print(r.stdout.strip() or r.stderr.strip()[-400:])

def show(fig):
    """Display a figure from results/figures/ inline."""
    display(Image(os.path.join(ROOT, "results", "figures", fig)))
from rsv import lineage

## 3.1  The lineages we analyse

Inclusion requires **at least 5 distinct years and at least 200 genomes**. The date window (2001 to the end of May 2026) and the sparse-end trimming are both applied inside `lineage.prepare()`, so no extra year filter is needed here.

In [ ]:
g = pd.read_csv("../data/processed/rsv_gene_features.tsv", sep="\t")
dates = pd.read_csv("../data/processed/rsv_features.tsv", sep="\t",
                    usecols=["Accession ID", "date"])
genome = g[g.gene == "GENOME"].merge(dates, on="Accession ID")
gdf, keep = lineage.prepare(genome)
print(f"{len(keep)} lineages, {len(gdf):,} genomes\n")
keep.round(2)

Two things to note:

- **The lineages do not cover the same calendar window.** B.D runs 2006–2016 while B.D.E runs 2018–2026, so they barely overlap. Comparisons between lineages are therefore partly comparisons between different periods.
- **RSV-B contributes three lineages, and one of them (B.D) is small** at 342 genomes. Any A-versus-B statement rests on a narrow base.

## 3.2  Trend of each feature within each lineage

Each feature is regressed against the **decimal collection date**, inside each lineage separately. The slope is the change per year.

In [ ]:
FEATS = ["gc_content", "gc3", "gc12",
         "per1k_CG", "per1k_GC", "per1k_GG", "per1k_CC", "per1k_TA",
         "oe_CG", "oe_GC", "oe_GG", "oe_CC", "oe_TA"]
slopes = lineage.lineage_slopes(gdf, FEATS)
slopes[slopes.feature == "gc_content"][
    ["Subtype", "lineage_grp", "n", "slope", "ci_lo", "ci_hi", "r2", "significant"]
].round(6)

### Figure — GC content over time, one panel per lineage

The equivalent of the paper's Figure 2a. Points are individual genomes; the line is the fitted trend with its confidence band.

In [ ]:
run_r()

In [ ]:
show("fig1_lineage_trends.png")

## 3.3  Where along the codon is GC changing?

The sharpest test available:

- pressure acting on the **RNA** → strongest effect at third codon positions, where changes are mostly synonymous
- pressure acting on the **protein** → first and second positions move comparably

In [ ]:
piv = (slopes[slopes.feature.isin(["gc_content", "gc3", "gc12"])]
       .pivot_table(index=["Subtype", "lineage_grp"], columns="feature", values="slope"))
piv["gc3 / gc12"] = (piv.gc3 / piv.gc12).round(2)
piv.round(6)

In [ ]:
show("fig2_gc3_vs_gc12.png")

### How this differs from the paper

| | influenza (paper) | RSV (here) |
|---|---|---|
| GC3 | falls | **rises in 6 of 9 lineages** |
| GC12 | falls | **falls in 6 of 9** |
| relationship | same direction, GC3 5.65× faster | **opposite directions**, ratio negative |

Two things follow:

- Synonymous and non-synonymous sites are clearly under different pressure, which is interesting in itself.
- This is **not** the influenza signature and should not be described as reproducing it.

Note the size: slopes are around 10⁻⁴ per year, so roughly a tenth of a percentage point of GC across the whole decade. Small.

## 3.4  The dinucleotides

The same regression applied to each dinucleotide, reported both as frequency per 1,000 nucleotides (the paper's unit) and as the O/E ratio. The O/E version matters because it already corrects for GC content, so a trend there is not simply GC content moving.

In [ ]:
cg = slopes[slopes.feature == "oe_CG"][
    ["Subtype", "lineage_grp", "n", "slope", "ci_lo", "ci_hi", "p", "significant"]]
cg.round(6)

In [ ]:
show("fig3_dinucleotide_slopes.png")

### Does geography explain these slopes?

Notebook 01 showed that where RSV was sequenced shifted over the period, including inside single lineages. Repeating the fit with **region held constant** tells us whether a slope is about time or about place.

In [ ]:
adj = lineage.lineage_slopes(gdf, ["oe_CG"], covariates=("region",))
cmp = (slopes[slopes.feature == "oe_CG"][["Subtype","lineage_grp","n","slope","p"]]
       .merge(adj[["Subtype","lineage_grp","slope","p"]],
              on=["Subtype","lineage_grp"], suffixes=("", "_adj")))
cmp["same_direction"] = (cmp.slope * cmp.slope_adj) > 0
cmp.round(6)

Most slopes keep their direction and rough size, so geography is not generating them. The ones that move most are the lineages notebook 01 flagged as having the largest geographic drift — the adjustment bites exactly where the problem is.

## 3.5  Is the dinucleotide signal just GC content moving?

The paper's Figure 3b. A dinucleotide that correlates almost perfectly with GC content carries no information beyond GC; correlations well below 1 mean it is moving on its own.

In [ ]:
corr = lineage.gc_vs_dinucleotide_correlation(
    gdf, ["per1k_CG", "per1k_GC", "per1k_GG", "per1k_CC", "per1k_TA"])
corr.groupby("feature").r.agg(["mean", "min", "max"]).round(3)

In [ ]:
show("fig4_gc_correlation.png")

## 3.6  Sensitivity: does the lineage level drive the result?

Truncating to three levels was a choice. Repeating at level 4 gives more, smaller lineages. If direction and rough size agree, the choice is not driving the conclusion.

In [ ]:
gdf4, keep4 = lineage.prepare(genome, level=4)
s4 = lineage.lineage_slopes(gdf4, ["gc_content", "gc3", "oe_CG"])
cmp = pd.concat([slopes[slopes.feature.isin(["gc_content","gc3","oe_CG"])].assign(level=3),
                 s4.assign(level=4)])
cmp.groupby(["level", "feature"]).slope.agg(
    median="median", lineages="size",
    n_negative=lambda s: int((s < 0).sum())).round(6)

**The CpG result does not survive this check.**

| Lineage level | median slope | lineages negative |
|---|---|---|
| 3 | slightly positive | 3 of 9 |
| 4 | negative | 15 of 20 |

The direction flips depending on how finely lineages are defined, which means the granularity is driving it rather than the biology.

GC3 is steadier — positive at both levels, though weaker at the finer one. GC content overall sits near zero either way.

---

## 3.7  What this notebook found

Stated plainly, because the result is mostly negative.

| | Finding |
|---|---|
| **1** | **No consistent CpG trend in RSV.** Lineages disagree: A.D.1 and B.D.E rise, A.D.3 and A.D.5 fall. In the influenza paper all twelve persistent lineages moved the same way. Here they do not, and section 3.6 shows the overall direction is an artefact of lineage definition. |
| **2** | **GC3 and GC12 move in opposite directions**, unlike influenza where both fell. Synonymous and non-synonymous sites are under different pressure in RSV, but not in the pattern the paper describes. |
| **3** | **Geography is not the explanation.** Slopes keep their direction after region adjustment, so the inconsistency between lineages is real, not a sampling artefact. |
| **4** | **All effects are small** — order 10⁻⁴ per year, a fraction of a percentage point across the study period. |

### How to read this

RSV has circulated in humans for a very long time and is already well adapted to it. The influenza result came from lineages that had recently entered a *new* host and were still adjusting. There is no equivalent transition inside human RSV, so finding no directional trend is a reasonable outcome rather than a failed analysis.

The clearest signal in this project is not temporal at all — it is the difference between genes and between the two subtypes, which notebook 04 covers.